# 5.2 코드 생성을 위한 RL


### 코드 생성 RL의 목표:
- **정의**: 주어진 문제 명세에서 정확한 코드를 생성하는 정책 최적화
- **특징**: 한 번의 생성 시도로 정답을 맞추는 확률 증가
- **성능 지표**: 정확한 코드 생성 확률 (0~100%)

**Generate → Execute → Score 패턴**:
```
1) Generate: LLM으로 코드 생성한다
2) Execute: exec()로 실행하고 오류 확인한다
3) Score: 테스트 케이스와 품질 평가한다
4) Reward: 점수를 정규화하여 정책 신호로 사용한다
```

## 코드 생성의 보상 함수

### 다층 보상 설계

```
보상 = 테스트통과도(50%) + 코드품질(30%) + 실행안정성(20%)

1) 테스트 통과도 (0~10)
   - 통과한 테스트 개수 / 전체 테스트 개수 * 10
   - 예: 3/5 통과 → 6점

2) 코드 품질 (0~10)
   - 가독성, 정확성, 효율성을 LLM으로 평가
   - 예: 명확한 변수명, 좋은 로직 → 8점

3) 실행 안정성 (0~10)
   - 오류 없이 실행되는가
   - 엣지 케이스 처리하는가
   - 예: 모든 테스트 케이스에서 실행 성공 → 10점
```

### REINFORCE 방식의 코드 생성 최적화

**아이디어**: 높은 보상을 받은 프롬프트와 코드 패턴을 더 자주 사용한다

```
정책 기울기:
  ∇ log π(y|x) × reward(y)

직관:
  - 높은 보상(reward > 0) → 그 코드를 생성할 확률 증가
  - 낮은 보상(reward < 0) → 그 코드를 생성할 확률 감소
  - 프롬프트도 함께 최적화 (프롬프트 GRPO)
```

In [1]:
from utils_openai import *
import numpy as np
import re

heading("준비: 코드 생성 RL")
print("✓ Generate: ask()로 코드 생성한다")
print("✓ Execute: exec()로 실행한다")
print("✓ Score: 테스트와 품질로 평가한다")
print("✓ Optimize: 보상으로 정책 개선한다")

def safe_llm_reward(text, criteria="정확성, 완전성, 유용성", max_score=10.0):
    """llm_reward의 안전한 래퍼이다. 숫자 파싱을 보장한다."""
    score_str = ask(
        f"평가 기준: {criteria}\n응답: {text}\n\n0~{max_score} 사이 숫자 하나만 반환하라. 설명 없이 숫자만 출력하라. 예: 7.5",
        temperature=0.0, max_tokens=5
    )
    numbers = re.findall(r'[\d.]+', score_str)
    return min(max(float(numbers[0]), 0.0), max_score) if numbers else max_score * 0.5

def strip_code_blocks(code_text):
    """LLM 응답에서 마크다운 코드 블록을 제거한다."""
    if "```" in code_text:
        code_text = code_text.replace("```python", "").replace("```", "").strip()
    return code_text


────────────────────────────────────────
  준비: 코드 생성 RL
────────────────────────────────────────
✓ Generate: ask()로 코드 생성한다
✓ Execute: exec()로 실행한다
✓ Score: 테스트와 품질로 평가한다
✓ Optimize: 보상으로 정책 개선한다


## 실습 1: Generate → Execute → Score 파이프라인

코드 생성 RL의 기본 루프를 구현한다.

In [2]:
heading("실습 1: Generate → Execute → Score")

problem = "파이썬으로 정수 n을 입력받아 팩토리얼을 반환하는 함수 factorial을 작성하라. n>=0이다."
test_cases = [(0, 1), (1, 1), (5, 120), (6, 720)]

step_print(1, "문제", problem)
step_print(2, "테스트 케이스", f"{len(test_cases)}개")
for n, expected in test_cases:
    print(f"  factorial({n}) = {expected}")

# Step 1: Generate
step_print(3, "Step 1: Generate", "LLM으로 코드 생성한다")
code = ask(
    problem,
    system="정확한 파이썬 함수를 작성한다. 설명 없이 함수 정의만 반환한다.",
    temperature=0.5
)
code = strip_code_blocks(code)
print(f"  생성된 코드 (길이: {len(code)} 글자):\n{code}")

# Step 2: Execute
step_print(4, "Step 2: Execute", "생성된 코드 실행한다")
namespace = {}
execution_success = True
test_results = []
factorial_func = None

exec(code, namespace)
factorial_func = namespace.get('factorial')

if factorial_func:
    print("  코드 실행 성공한다")
    for n, expected in test_cases:
        result = factorial_func(n)
        passed = result == expected
        test_results.append(passed)
        status = "통과" if passed else "실패"
        print(f"  테스트 {n}: {result} (기대: {expected}) [{status}]")
else:
    execution_success = False
    print("  함수 'factorial'을 찾을 수 없다")

# Step 3: Score
step_print(5, "Step 3: Score", "다층 보상 계산한다")

if execution_success and factorial_func and test_results:
    test_pass_rate = sum(test_results) / len(test_results)
    test_score = test_pass_rate * 10
    print(f"  테스트 통과도: {sum(test_results)}/{len(test_results)} = {test_pass_rate:.1%} → {test_score:.1f}/10")

    quality_score = safe_llm_reward(code, criteria="파이썬 함수의 정확성, 간결함, 가독성", max_score=10.0)
    print(f"  코드 품질: {quality_score:.1f}/10")

    stability_score = 10.0 if all(test_results) else 5.0
    print(f"  실행 안정성: {stability_score:.1f}/10")

    final_reward = test_score * 0.5 + quality_score * 0.3 + stability_score * 0.2
    print(f"\n  최종 보상: {final_reward:.2f}/10")
else:
    print("  실행 실패로 점수 계산 불가능하다")


────────────────────────────────────────
  실습 1: Generate → Execute → Score
────────────────────────────────────────
  [Step 1] 문제
    파이썬으로 정수 n을 입력받아 팩토리얼을 반환하는 함수 factorial을 작성하라. n>=0이다.
  [Step 2] 테스트 케이스
    4개
  factorial(0) = 1
  factorial(1) = 1
  factorial(5) = 120
  factorial(6) = 720
  [Step 3] Step 1: Generate
    LLM으로 코드 생성한다
  생성된 코드 (길이: 95 글자):
def factorial(n):
    if n == 0:
        return 1
    else:
        return n * factorial(n - 1)
  [Step 4] Step 2: Execute
    생성된 코드 실행한다
  코드 실행 성공한다
  테스트 0: 1 (기대: 1) [통과]
  테스트 1: 1 (기대: 1) [통과]
  테스트 5: 120 (기대: 120) [통과]
  테스트 6: 720 (기대: 720) [통과]
  [Step 5] Step 3: Score
    다층 보상 계산한다
  테스트 통과도: 4/4 = 100.0% → 10.0/10
  코드 품질: 7.0/10
  실행 안정성: 10.0/10

  최종 보상: 9.10/10


## 실습 2: 여러 프롬프트 전략 비교

다양한 프롬프트로 코드를 생성하고 성능을 비교한다.

In [3]:
heading("실습 2: 프롬프트 전략 비교")

strategies = {
    "기본": "파이썬 함수를 작성하라. 두 수의 합을 반환한다. def add(a, b):",
    "상세": "파이썬 함수를 작성하라. 두 수의 합을 반환한다. 타입 힌팅을 포함하라. 주석을 추가하라. def add(a, b):",
    "단계형": "파이썬 함수를 단계적으로 작성하라. 1) 함수 정의 2) 입력 매개변수 3) 덧셈 로직 4) 반환. def add(a, b):",
    "예시형": "파이썬 함수를 작성하라. 두 수의 합을 반환한다. 예: add(2, 3) = 5. def add(a, b):"
}

step_print(1, "프롬프트 전략", list(strategies.keys()))

strategy_scores = {}

for strategy_name, prompt in strategies.items():
    print(f"\n  [{strategy_name}]")
    code = ask(prompt, system="설명 없이 함수만 반환한다.", temperature=0.3)
    code = strip_code_blocks(code)

    namespace = {}
    score = 0
    exec(code, namespace)
    add_func = namespace.get('add')
    if add_func and add_func(2, 3) == 5:
        score = 9.0
    else:
        score = 5.0

    strategy_scores[strategy_name] = score
    print(f"    점수: {score:.1f}/10")

step_print(2, "비교 결과", "최고 성능 전략")
best_strategy = max(strategy_scores, key=strategy_scores.get)
print(f"  최고: {best_strategy} ({strategy_scores[best_strategy]:.1f}/10)")
print(f"  → 이 전략을 더 자주 사용한다 (REINFORCE)")


────────────────────────────────────────
  실습 2: 프롬프트 전략 비교
────────────────────────────────────────
  [Step 1] 프롬프트 전략
    • 기본
    • 상세
    • 단계형
    • 예시형

  [기본]
    점수: 9.0/10

  [상세]
    점수: 9.0/10

  [단계형]
    점수: 9.0/10

  [예시형]
    점수: 9.0/10
  [Step 2] 비교 결과
    최고 성능 전략
  최고: 기본 (9.0/10)
  → 이 전략을 더 자주 사용한다 (REINFORCE)


## 실습 3: 다양한 코딩 문제 유형별 RL

다양한 종류의 코딩 문제에 RL을 적용한다.

In [4]:
heading("실습 3: 코딩 문제 유형별 RL")

problem_types = {
    "수학": {
        "문제": "두 정수의 최대공약수를 구하는 함수 gcd를 작성하라.",
        "테스트": [(12, 8, 4), (100, 50, 50)]
    },
    "문자열": {
        "문제": "문자열을 입력받아 가장 많이 나타나는 문자를 반환하는 함수를 작성하라.",
        "테스트": [("aabbcc", "a")]
    },
    "배열": {
        "문제": "배열의 모든 원소를 정렬된 순서로 반환하는 함수를 작성하라.",
        "테스트": [([3,1,2], [1,2,3])]
    }
}

step_print(1, "문제 유형", list(problem_types.keys()))

for ptype, pdata in problem_types.items():
    print(f"\n  [{ptype}]")
    print(f"    문제: {pdata['문제'][:50]}...")
    print(f"    테스트 케이스: {len(pdata['테스트'])}개")
    print(f"    예상 성능: {'높다' if ptype == '수학' else '중간'}")

step_print(2, "성능 분석", "문제 유형별 난이도")
print(f"  수학 문제: 알고리즘 명확 → 높은 성공률")
print(f"  문자열 문제: 엣지 케이스 많음 → 중간 성공률")
print(f"  배열 문제: 여러 구현 방법 → 중간 성공률")


────────────────────────────────────────
  실습 3: 코딩 문제 유형별 RL
────────────────────────────────────────
  [Step 1] 문제 유형
    • 수학
    • 문자열
    • 배열

  [수학]
    문제: 두 정수의 최대공약수를 구하는 함수 gcd를 작성하라....
    테스트 케이스: 2개
    예상 성능: 높다

  [문자열]
    문제: 문자열을 입력받아 가장 많이 나타나는 문자를 반환하는 함수를 작성하라....
    테스트 케이스: 1개
    예상 성능: 중간

  [배열]
    문제: 배열의 모든 원소를 정렬된 순서로 반환하는 함수를 작성하라....
    테스트 케이스: 1개
    예상 성능: 중간
  [Step 2] 성능 분석
    문제 유형별 난이도
  수학 문제: 알고리즘 명확 → 높은 성공률
  문자열 문제: 엣지 케이스 많음 → 중간 성공률
  배열 문제: 여러 구현 방법 → 중간 성공률


## 실습 4: 함수 생성 → 테스트 → 보상 파이프라인

전체 파이프라인을 한 번에 실행한다.

In [5]:
heading("실습 4: 완전 파이프라인")

problem = "파이썬으로 문자열이 회문(palindrome)인지 판단하는 함수 is_palindrome을 작성하라. 대소문자 무시한다."
test_cases = [
    ("A", True),
    ("Aa", True),
    ("racecar", True),
    ("hello", False)
]

step_print(1, "파이프라인 시작", problem)

# 1. 생성
step_print(2, "1) 생성", "ask()로 코드 생성한다")
code = ask(
    problem,
    system="정확한 파이썬 함수를 작성한다. 설명 없이 함수만 반환한다.",
    temperature=0.4
)
code = strip_code_blocks(code)
print(f"  생성 완료: {len(code)} 글자")

# 2. 실행
step_print(3, "2) 실행", "exec()로 함수 실행한다")
namespace = {}
test_passed = 0
test_total = len(test_cases)

exec(code, namespace)
is_palindrome = namespace.get('is_palindrome')

if is_palindrome:
    print("  함수 로드 성공한다")
    for test_input, expected in test_cases:
        result = is_palindrome(test_input)
        if result == expected:
            test_passed += 1
            print(f"  is_palindrome('{test_input}') = {result} ✓")
        else:
            print(f"  is_palindrome('{test_input}') = {result} (기대: {expected}) ✗")
else:
    print("  함수 로드 실패한다")

# 3. 평가
step_print(4, "3) 평가", "보상 계산한다")
test_rate = test_passed / test_total
test_reward = test_rate * 10.0
quality_reward = safe_llm_reward(code, criteria="파이썬 코드 정확성과 간결함", max_score=10.0)
final_score = test_reward * 0.6 + quality_reward * 0.4

print(f"  테스트 통과율: {test_passed}/{test_total} = {test_rate:.0%}")
print(f"  테스트 보상: {test_reward:.1f}/10")
print(f"  품질 보상: {quality_reward:.1f}/10")
print(f"  최종 점수: {final_score:.1f}/10")
print(f"  결과: {'성공' if final_score >= 8 else '양호' if final_score >= 6 else '개선 필요'}")


────────────────────────────────────────
  실습 4: 완전 파이프라인
────────────────────────────────────────
  [Step 1] 파이프라인 시작
    파이썬으로 문자열이 회문(palindrome)인지 판단하는 함수 is_palindrome을 작성하라. 대소문자 무시한다.
  [Step 2] 1) 생성
    ask()로 코드 생성한다
  생성 완료: 63 글자
  [Step 3] 2) 실행
    exec()로 함수 실행한다
  함수 로드 성공한다
  is_palindrome('A') = True ✓
  is_palindrome('Aa') = True ✓
  is_palindrome('racecar') = True ✓
  is_palindrome('hello') = False ✓
  [Step 4] 3) 평가
    보상 계산한다
  테스트 통과율: 4/4 = 100%
  테스트 보상: 10.0/10
  품질 보상: 9.0/10
  최종 점수: 9.6/10
  결과: 성공


## 요약: 코드 생성을 위한 RL

### Generate → Execute → Score 패턴

1. **Generate**: 프롬프트로 코드 생성한다
   - 온도 조정: 낮음(0.3)은 정확, 높음(0.8)은 다양
   - 프롬프트 최적화: 상세함 vs 간결함 트레이드오프

2. **Execute**: 코드 실행과 테스트한다
   - exec()로 함수 정의한다
   - 테스트 케이스로 검증한다
   - 오류 메시지 수집한다

3. **Score**: 다층 보상 계산한다
   - 테스트 통과도 (50%)
   - 코드 품질 (30%)
   - 실행 안정성 (20%)

### REINFORCE 기반 최적화

- **정책**: 주어진 프롬프트에서 어떤 코드를 생성할 확률
- **보상 신호**: 테스트 통과율과 코드 품질
- **업데이트**: 높은 보상 코드를 더 자주 생성하도록 학습

### 성능 향상 전략

1. **더 명확한 프롬프트**: 예시와 단계 포함한다
2. **더 좋은 테스트**: 엣지 케이스 포함한다
3. **더 나은 기본 모델**: GPT-4 사용한다
4. **반복 시도**: Best-of-N으로 여러 번 생성한다